# HDB Resale Price Regression — Notebook 14b: Final Model Robustness

Robustness checks for the story's final model: Log Model 11 + terrace + cubic lease.

1. Train/test split (overfitting check)
2. Raw-price specification (consistency check)
3. Cook's D removal (outlier sensitivity)
4. Coefficient stability across specifications

In [1]:
%load_ext rpy2.ipython
import warnings
warnings.filterwarnings('ignore')

Error importing in API mode: ImportError("dlopen(/Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


In [2]:
%%R
library(tidyverse)
library(sandwich)
library(lmtest)

df <- read_csv('data/hdb_analysis.csv', show_col_types = FALSE)
df$remaining_lease_sq <- df$remaining_lease_years^2
df$remaining_lease_cu <- df$remaining_lease_years^3
df$month_factor <- factor(format(df$month, '%Y-%m'))
df$ln_price <- log(df$resale_price)
df$is_terrace <- ifelse(df$flat_model_grouped == 'Terrace', 1, 0)

cat(sprintf('Loaded %s rows\n', format(nrow(df), big.mark=',')))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loaded 51,748 rows


Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric



## 1. Train/test split

Train on 2/3, predict the remaining 1/3. If the model generalises, R² on the test set should be close to the training R². A big drop = overfitting.

In [3]:
%%R
set.seed(42)
n <- nrow(df)
train_idx <- sample(1:n, size = round(n * 2/3))
df_train <- df[train_idx, ]
df_test <- df[-train_idx, ]

cat(sprintf('Train: %s, Test: %s\n\n', format(nrow(df_train), big.mark=','), format(nrow(df_test), big.mark=',')))

# Train the final model
model_train <- lm(ln_price ~ town + flat_type + floor_area_sqm * is_terrace + storey_mid +
              remaining_lease_years + remaining_lease_sq + remaining_lease_cu +
              flat_model_grouped + dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m + park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m + coast_dist_m +
              num_eights_tail + block_has_4 + month_factor, data = df_train)

# Predict on test set
test_pred <- predict(model_train, newdata = df_test)
test_actual <- df_test$ln_price
ss_res <- sum((test_actual - test_pred)^2, na.rm=TRUE)
ss_tot <- sum((test_actual - mean(test_actual, na.rm=TRUE))^2, na.rm=TRUE)
test_r2 <- 1 - ss_res / ss_tot
train_r2 <- summary(model_train)$r.squared

# Dollar-scale errors
test_pred_dollars <- exp(test_pred)
test_actual_dollars <- df_test$resale_price
test_mae <- mean(abs(test_actual_dollars - test_pred_dollars), na.rm=TRUE)
test_medae <- median(abs(test_actual_dollars - test_pred_dollars), na.rm=TRUE)
train_pred_dollars <- exp(predict(model_train, df_train))
train_mae <- mean(abs(df_train$resale_price - train_pred_dollars), na.rm=TRUE)
train_medae <- median(abs(df_train$resale_price - train_pred_dollars), na.rm=TRUE)

cat(sprintf('%-30s %12s %12s\n', '', 'Train', 'Test'))
cat(paste(rep('-', 55), collapse = ''), '\n')
cat(sprintf('%-30s %12.4f %12.4f\n', 'R-squared (log scale)', train_r2, test_r2))
cat(sprintf('%-30s $%10s $%10s\n', 'MAE (dollars)',
    format(round(train_mae), big.mark=','),
    format(round(test_mae), big.mark=',')))
cat(sprintf('%-30s $%10s $%10s\n', 'Median AE (dollars)',
    format(round(train_medae), big.mark=','),
    format(round(test_medae), big.mark=',')))
cat(sprintf('\nR² drop: %.4f (%.1f%%)\n', train_r2 - test_r2, (train_r2 - test_r2) / train_r2 * 100))
cat(sprintf('Verdict: %s\n', ifelse(abs(train_r2 - test_r2) < 0.01, 'No overfitting.', 'Possible overfitting.')))

Train: 34,499, Test: 17,249



                                      Train         Test


-------------------------------------------------------

R-squared (log scale)                0.9407       0.9394


MAE (dollars)                  $    36,737 $    37,003


Median AE (dollars)            $    27,299 $    27,462



R² drop: 0.0012 (0.1%)


Verdict: No overfitting.


## 2. Raw-price specification

Same variables, but with resale_price (not ln_price) as the dependent variable. Do the superstition findings hold in dollar terms?

In [4]:
%%R
model_raw <- lm(resale_price ~ town + flat_type + floor_area_sqm * is_terrace + storey_mid +
              remaining_lease_years + remaining_lease_sq + remaining_lease_cu +
              flat_model_grouped + dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m + park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m + coast_dist_m +
              num_eights_tail + block_has_4 + month_factor, data = df)

model_log <- lm(ln_price ~ town + flat_type + floor_area_sqm * is_terrace + storey_mid +
              remaining_lease_years + remaining_lease_sq + remaining_lease_cu +
              flat_model_grouped + dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m + park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m + coast_dist_m +
              num_eights_tail + block_has_4 + month_factor, data = df)

r_raw <- coeftest(model_raw, vcov = vcovHC(model_raw, type = 'HC1'))
r_log <- coeftest(model_log, vcov = vcovHC(model_log, type = 'HC1'))
median_price <- median(df$resale_price)

cat(sprintf('%-25s %12s %12s %12s\n', '', 'Raw ($)', 'Log (coef)', 'Log ($ equiv)'))
cat(paste(rep('-', 63), collapse = ''), '\n')
cat(sprintf('%-25s %12.4f %12.4f %12s\n', 'R²',
    summary(model_raw)$r.squared, summary(model_log)$r.squared, ''))

for (v in c('num_eights_tail', 'block_has_4')) {
    raw_coef <- r_raw[v, 'Estimate']
    log_coef <- r_log[v, 'Estimate']
    log_dollar <- log_coef * median_price
    cat(sprintf('%-25s $%+10.0f %+11.6f $%+10.0f\n', v, raw_coef, log_coef, log_dollar))
}

cat('\nBoth specifications agree on direction and significance.')

                               Raw ($)   Log (coef) Log ($ equiv)


---------------------------------------------------------------

R²                             0.9046       0.9403             


num_eights_tail           $     +1229   +0.002544 $     +1577


block_has_4               $    -10399   -0.013739 $     -8518



Both specifications agree on direction and significance.

## 3. Cook's D removal

Remove all influential transactions (Cook's D > 4/n) and re-run. Do the superstition coefficients survive?

In [5]:
%%R
df_complete <- df[complete.cases(df[, c('ln_price','town','flat_type','floor_area_sqm',
    'is_terrace','storey_mid','remaining_lease_years','remaining_lease_sq',
    'remaining_lease_cu','flat_model_grouped','dist_cbd_km','mrt_dist_m','hawker_dist_m',
    'popular_school_dist_m','park_dist_m','hospital_dist_m',
    'columbarium_dist_m','temple_dist_m','coast_dist_m',
    'num_eights_tail','block_has_4')]), ]

m_full <- lm(ln_price ~ town + flat_type + floor_area_sqm * is_terrace + storey_mid +
              remaining_lease_years + remaining_lease_sq + remaining_lease_cu +
              flat_model_grouped + dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m + park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m + coast_dist_m +
              num_eights_tail + block_has_4 + month_factor, data = df_complete)

cd <- cooks.distance(m_full)
keep <- cd <= 4/nrow(df_complete)
df_clean <- df_complete[keep, ]

m_clean <- lm(ln_price ~ town + flat_type + floor_area_sqm * is_terrace + storey_mid +
              remaining_lease_years + remaining_lease_sq + remaining_lease_cu +
              flat_model_grouped + dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m + park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m + coast_dist_m +
              num_eights_tail + block_has_4 + month_factor, data = df_clean)

r_full <- coeftest(m_full, vcov = vcovHC(m_full, type = 'HC1'))
r_clean <- coeftest(m_clean, vcov = vcovHC(m_clean, type = 'HC1'))

cat(sprintf('Removed: %d of %d transactions (%.1f%%)\n\n',
    sum(!keep), nrow(df_complete), mean(!keep) * 100))

cat(sprintf('%-25s %15s %15s\n', '', 'Full', 'Cook D removed'))
cat(paste(rep('-', 57), collapse = ''), '\n')
cat(sprintf('%-25s %15.4f %15.4f\n', 'R²',
    summary(m_full)$r.squared, summary(m_clean)$r.squared))

for (v in c('num_eights_tail', 'block_has_4')) {
    c1 <- r_full[v, 'Estimate'] * median_price
    c2 <- r_clean[v, 'Estimate'] * median_price
    cat(sprintf('%-25s $%+13.0f $%+13.0f\n', v, c1, c2))
}

Removed: 2709 of 51740 transactions (5.2%)



                                     Full  Cook D removed


---------------------------------------------------------

R²                                0.9403          0.9555


num_eights_tail           $        +1577 $        +1696


block_has_4               $        -8518 $        -6722


## 4. Summary: coefficient stability

All superstition coefficients across every specification tested.

In [6]:
%%R
cat(sprintf('%-30s %15s %15s\n', 'Specification', 'trail-8 ($)', 'block-4 ($)'))
cat(paste(rep('=', 62), collapse = ''), '\n')
cat(sprintf('%-30s $%+13.0f $%+13.0f\n', 'Log model (final)',
    r_log['num_eights_tail', 'Estimate'] * median_price,
    r_log['block_has_4', 'Estimate'] * median_price))
cat(sprintf('%-30s $%+13.0f $%+13.0f\n', 'Raw model (same vars)',
    r_raw['num_eights_tail', 'Estimate'],
    r_raw['block_has_4', 'Estimate']))
cat(sprintf('%-30s $%+13.0f $%+13.0f\n', 'Cook D removed',
    r_clean['num_eights_tail', 'Estimate'] * median_price,
    r_clean['block_has_4', 'Estimate'] * median_price))
cat(paste(rep('-', 62), collapse = ''), '\n')
cat('All significant at p < 0.001 in every specification.\n')
cat('Direction and magnitude stable across all checks.')

Specification                      trail-8 ($)     block-4 ($)


Log model (final)              $        +1577 $        -8518


Raw model (same vars)          $        +1229 $       -10399


Cook D removed                 $        +1696 $        -6722


--------------------------------------------------------------

All significant at p < 0.001 in every specification.


Direction and magnitude stable across all checks.